# Project - Progress Report #2: Regression
## Cybersecurity Vulnerability Severity Prediction Using Graph Machine Learning

**Team:** Chesda Saroth, Tabitha Ondieki, Abdullahi Oyeniyi  
**Data source:** Unified Cybersecurity Knowledge Graph (UCKG) CVE sample  
**Repository:** `Abdullahi023/Data-Science-Project`  
**Deliverable:** `code/regression.ipynb`  
**Date:** November 2026

### Objectives

1. Fit a **simple linear regression** using two numerical attributes and visualize observations versus predictions.
2. Fit a **multiple linear regression** using more than two numerical predictors and plot results with `mplot3d`.
3. Fit a **logistic regression** to predict CVE severity from CVSS vector attributes and visualize predictions versus observations.
4. Compare held-out performance, explain limitations, and summarize what was learned.

The assignment's reference to a *consumable usage dataset* is treated as a generic example: our previously selected dataset contains **cybersecurity vulnerabilities**, not consumable usage. Models use a held-out test set to reduce optimistic evaluation. We do not invent performance values.

> **Reproducibility:** Place this notebook in `code/` and keep `cve_clean.parquet` in `dataset/`. Execute all cells from top to bottom. Install required packages in the environment first: `pip install pandas numpy pyarrow matplotlib scikit-learn ipykernel` (the existing repository `req.txt` may need `scikit-learn` added). Once executed on the real data, re-export the notebook to PDF to include actual output figures and scores.

## 0. Import libraries and load the dataset

We import Pandas and NumPy for data processing, `matplotlib` (including `mplot3d`) for plots, and scikit-learn for the models, train/test splits, preprocessing, and evaluation. We set a random seed so the plots and data split are reproducible.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # registers 3D projection
from IPython.display import display, Markdown

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_absolute_error,
                             mean_squared_error, accuracy_score,
                             balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True})
print("Libraries imported successfully.")

**Interpretation:** `LinearRegression` predicts a continuous quantity; `LogisticRegression` predicts one of several discrete severity categories. We report **test-set** metrics rather than training-set scores.

### Load the cleaned CVE sample

The intended location is `../dataset/cve_clean.parquet` when Jupyter's working directory is `code/`. The alternative `dataset/cve_clean.parquet` supports launching from the repository root. We verify the expected fields and count the severity labels before modeling.

In [ ]:
paths = [Path("../dataset/cve_clean.parquet"), Path("dataset/cve_clean.parquet")]
dataset_path = next((p for p in paths if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Could not find cve_clean.parquet. Keep it in the project's dataset/ folder "
        "and run this notebook from the code/ folder or repository root."
    )

df = pd.read_parquet(dataset_path)
required = {"description", "severity", "cvss_vector"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing dataset columns: {sorted(missing)}")

print("Loaded:", dataset_path.resolve())
print("Rows and columns:", df.shape)
print("Original fields:", list(df.columns))
print("Severity distribution:")
display(df["severity"].astype("string").str.upper().value_counts().to_frame("count"))
display(df[["description", "severity", "cvss_vector"]].head(5))

**Interpretation:** The team's prior exploration identified **909 cleaned CVEs** (504 Medium, 322 High, 83 Low; no Critical examples at that time). The printed table above verifies the dataset actually loaded, so any updated dataset will be reflected in the analysis without hard-coding its size.

### Engineer numerical attributes from the vulnerability descriptions

The original data includes free-text `description` values, so we calculate numerical characteristics. These text counts are suitable for demonstrating linear regression. We independently derive alphabetic, digit, whitespace and other-character counts; by definition they add up to total description length.

In [ ]:
model_df = df.copy()
model_df["description"] = model_df["description"].fillna("").astype(str)
model_df = model_df.loc[model_df["description"].str.strip().ne("")].copy()

model_df["description_length"] = model_df["description"].str.len()
model_df["word_count"] = model_df["description"].str.split().str.len()
model_df["letter_count"] = model_df["description"].str.count(r"[A-Za-z]")
model_df["digit_count"] = model_df["description"].str.count(r"\d")
model_df["whitespace_count"] = model_df["description"].str.count(r"\s")
model_df["other_character_count"] = (
    model_df["description_length"]
    - model_df["letter_count"]
    - model_df["digit_count"]
    - model_df["whitespace_count"]
)

feature_columns = ["description_length", "word_count", "letter_count",
                   "digit_count", "whitespace_count", "other_character_count"]
print("Numerical attributes created:")
display(model_df[feature_columns].describe().round(2))
assert (model_df["description_length"] ==
        model_df[["letter_count", "digit_count", "whitespace_count",
                  "other_character_count"]].sum(axis=1)).all()

**Interpretation:** The summary displays the typical lengths and counts in the CVE narratives. The assertion checks that the four character categories exactly reconstruct the full description length. This matters for Section 2: perfect reconstruction is **not** evidence of generalizable prediction.

## 1. Simple Linear Regression (two attributes)

**Predictor (X):** `description_length` (number of characters)  
**Target (y):** `word_count` (number of words)

We fit `sklearn.linear_model.LinearRegression` on **80% training / 20% testing** data. R² measures explained test-set variance; mean absolute error (MAE) and root mean squared error (RMSE) measure the size of prediction errors in **words**. Description length and word count should be positively related, but this relationship is about text structure, not vulnerability severity.

In [ ]:
X_simple = model_df[["description_length"]]
y_simple = model_df["word_count"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.20, random_state=SEED
)

simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train_s)
simple_pred = simple_model.predict(X_test_s)

simple_r2 = r2_score(y_test_s, simple_pred)
simple_mae = mean_absolute_error(y_test_s, simple_pred)
simple_rmse = np.sqrt(mean_squared_error(y_test_s, simple_pred))

print(f"Equation: predicted words = {simple_model.coef_[0]:.4f} * characters "
      f"+ {simple_model.intercept_:.4f}")
print(f"Test R²   : {simple_r2:.4f}")
print(f"Test MAE  : {simple_mae:.2f} words")
print(f"Test RMSE : {simple_rmse:.2f} words")

**Interpretation:** A higher **test R²** (closer to 1) indicates a better fit to word counts; **lower MAE/RMSE** indicate smaller errors. The positive slope represents additional expected words as descriptions get longer. Treat the reported test score as the actual finding; no score is presumed before execution.

### Plot observed and predicted word counts with matplotlib

The blue points are the observed test examples, and the orange regression line gives the model's predictions at each description length. A narrow scatter around the line indicates a strong linear relationship.

In [ ]:
order_s = np.argsort(X_test_s["description_length"].to_numpy())
x_s = X_test_s["description_length"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(x_s, y_test_s, alpha=0.48, s=22, label="Observed word count")
ax.plot(x_s[order_s], simple_pred[order_s], color="darkorange", lw=2.3,
        label="Predicted word count")
ax.set(title=f"Simple linear regression (test R² = {simple_r2:.3f})",
       xlabel="Description length (characters)", ylabel="Word count")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** The fitted line describes the **average** word count for a given character length, not every example perfectly. Large vertical gaps between blue points and orange predictions are residual errors caused by differences in wording and word length.

## 2. Multiple Linear Regression (more than two predictors)

**Target (y):** `description_length`  
**Four predictors (X):** `letter_count`, `digit_count`, `whitespace_count`, `other_character_count`

We again use `sklearn.linear_model.LinearRegression` and an 80/20 test split. This is a deliberately transparent **arithmetic sanity check**: the four counts partition the text into mutually exclusive character categories and therefore sum to description length. We expect extremely high R² even on held-out data; that performance cannot be used as evidence of a useful real-world predictive model.

In [ ]:
multiple_features = ["letter_count", "digit_count", "whitespace_count",
                     "other_character_count"]
X_multi = model_df[multiple_features]
y_multi = model_df["description_length"]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.20, random_state=SEED
)

multiple_model = LinearRegression()
multiple_model.fit(X_train_m, y_train_m)
multi_pred = multiple_model.predict(X_test_m)

multi_r2 = r2_score(y_test_m, multi_pred)
multi_mae = mean_absolute_error(y_test_m, multi_pred)
multi_rmse = np.sqrt(mean_squared_error(y_test_m, multi_pred))

print("Coefficients:")
for feature, coefficient in zip(multiple_features, multiple_model.coef_):
    print(f"  {feature:<25} {coefficient:.4f}")
print(f"Intercept : {multiple_model.intercept_:.5f}")
print(f"Test R²   : {multi_r2:.4f}")
print(f"Test MAE  : {multi_mae:.6f} characters")
print(f"Test RMSE : {multi_rmse:.6f} characters")

**Interpretation:** The exact arithmetic identity implies coefficients near **1**, intercept near **0**, R² near **1**, and errors near **0**. This meets the exercise's high-performance demonstration but should be described honestly as **target reconstruction**: it would be misleading to call it a successful forecasting model. The test set confirms the formula works, not that independent information predicts a new outcome.

### Visualize observed and predicted values using mplot3d

The model contains four predictors, which cannot all be drawn on three axes. We project two (`letter_count` and `whitespace_count`) onto horizontal axes; the vertical axis is the target (`description_length`). Each test case appears once as an observed point and once as its predicted point. Other predictors remain part of the trained model even though they are not explicitly graphed.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_test_m["letter_count"], X_test_m["whitespace_count"],
           y_test_m, c="royalblue", s=20, alpha=0.55, label="Observed")
ax.scatter(X_test_m["letter_count"], X_test_m["whitespace_count"],
           multi_pred, c="darkorange", s=15, alpha=0.65,
           marker="^", label="Predicted")
ax.set_xlabel("Letter count")
ax.set_ylabel("Whitespace count")
ax.set_zlabel("Description length (characters)")
ax.set_title(f"Multiple linear regression (test R² = {multi_r2:.3f})")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation:** The predicted markers should nearly overlap observed points because the target is constructed directly from these character counts. Note that a 3D projection hides the digit and other-character axes, so two points at similar plotted positions can have different total lengths.

## 3. Logistic Regression for CVE severity

**Target:** `severity` (LOW, MEDIUM, HIGH, and CRITICAL if present)  
**Predictors:** individual CVSS-vector attributes (`AV`, `AC`, `AU`, `PR`, `UI`, `S`, `C`, `I`, `A`) and the vector's version.

CVSS vectors contain structured values such as `AV:N`, `AC:L`, or `C:P`. We parse them into categorical predictors instead of trying to use raw strings as numbers. `OneHotEncoder` converts each category into an indicator and `PolynomialFeatures` adds pairwise indicator interactions; finally `sklearn.linear_model.LogisticRegression` learns the multiclass boundaries. The test split is **stratified** so each severity class is represented fairly. Evaluation includes accuracy, balanced accuracy, macro-F1, per-class scores, and a majority-class baseline.

**Scientific limitation:** The published CVSS severity class is derived largely from the same CVSS vector metrics. Thus these are strong but **label-adjacent inputs**, not independent measures of future exploitation or risk. Results measure whether the model learns the known CVSS-to-severity mapping; they should not be presented as novel threat prediction.

In [ ]:
cvss_keys = ["AV", "AC", "AU", "PR", "UI", "S", "C", "I", "A"]

def parse_cvss(value):
    """Return a consistent dict for CVSS v2 and v3 (and partial v4) vectors."""
    value = "" if pd.isna(value) else str(value).strip()
    parts = value.split("/")
    result = {key: "MISSING" for key in cvss_keys}
    result["cvss_version"] = "3.x" if value.startswith("CVSS:3") else (
        "4.x" if value.startswith("CVSS:4") else "2.x/unknown"
    )
    for part in parts:
        if ":" not in part:
            continue
        key, category = part.split(":", 1)
        key = key.strip().upper()
        if key in result and category.strip():
            result[key] = category.strip().upper()
    return result

cvss_encoded = pd.DataFrame(
    model_df["cvss_vector"].apply(parse_cvss).tolist(),
    index=model_df.index
)
severity_values = model_df["severity"].fillna("").astype(str).str.strip().str.upper()
valid_severity = severity_values.isin(["LOW", "MEDIUM", "HIGH", "CRITICAL"])
X_logistic = cvss_encoded.loc[valid_severity].copy()
y_logistic = severity_values.loc[valid_severity].copy()

print("Severity examples:")
display(y_logistic.value_counts().to_frame("count"))
print("Parsed CVSS attributes:")
display(X_logistic.head())
if y_logistic.nunique() < 2 or y_logistic.value_counts().min() < 2:
    raise ValueError("Need at least two severity categories with at least two rows each.")

**Interpretation:** This processing keeps each CVSS impact or access metric distinct. Missing vector values receive the explicit category `MISSING`; no severity information is inserted into the input predictors. The printed distributions also expose any class imbalance.

### Train and evaluate logistic regression on a stratified holdout

The model uses a **20% test set** reserved before encoding or fitting. Pairwise interactions make a linear classifier more flexible for the nonlinear CVSS score rules. The majority-class baseline is included because a dataset dominated by Medium cases can produce deceptively high accuracy without a good classifier.

In [ ]:
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_logistic, y_logistic, test_size=0.20, random_state=SEED,
    stratify=y_logistic
)

logistic_model = Pipeline(steps=[
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ("pairwise_interactions", PolynomialFeatures(
        degree=2, interaction_only=True, include_bias=False)),
    ("classifier", LogisticRegression(max_iter=5000, C=10.0, solver="lbfgs"))
])
logistic_model.fit(X_train_l, y_train_l)
logistic_pred = logistic_model.predict(X_test_l)

logistic_accuracy = accuracy_score(y_test_l, logistic_pred)
logistic_bal_acc = balanced_accuracy_score(y_test_l, logistic_pred)
logistic_macro_f1 = f1_score(y_test_l, logistic_pred, average="macro", zero_division=0)
majority_baseline = y_test_l.value_counts(normalize=True).max()

print(f"Training observations: {len(y_train_l)} | Test observations: {len(y_test_l)}")
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")
print(f"Test accuracy                 : {logistic_accuracy:.3f}")
print(f"Test balanced accuracy        : {logistic_bal_acc:.3f}")
print(f"Test macro-F1                 : {logistic_macro_f1:.3f}")
print("\nPer-class precision, recall, and F1:")
print(classification_report(y_test_l, logistic_pred, zero_division=0))

**Interpretation:** Accuracy is the fraction of correctly labeled CVEs, while **macro-F1** averages the F1 score for each severity class without giving the most common class extra influence. Balanced accuracy averages per-class recall. Performance should be evaluated against the baseline printed above; strong test scores are plausible because the CVSS attributes themselves define much of severity, but no numerical score is claimed in advance.

### Plot observed versus predicted severity with matplotlib

The left panel compares the actual and predicted labels for individual test records, with small vertical jitter so overlapping markers remain visible. The right panel is a confusion matrix; strong classification produces a concentration along its diagonal. Only severity classes found in the dataset are displayed.

In [ ]:
severity_order = [v for v in ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
                  if v in set(y_logistic)]
severity_map = {value: i for i, value in enumerate(severity_order)}
actual_codes = y_test_l.map(severity_map).to_numpy()
predicted_codes = pd.Series(logistic_pred).map(severity_map).to_numpy()
idx = np.arange(len(actual_codes))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4),
                         gridspec_kw={"width_ratios": [1.15, 1]})
axes[0].scatter(idx, actual_codes + 0.06, s=23, alpha=0.55,
                label="Observed", marker="o")
axes[0].scatter(idx, predicted_codes - 0.06, s=20, alpha=0.55,
                label="Predicted", marker="x")
axes[0].set(yticks=range(len(severity_order)), yticklabels=severity_order,
            xlabel="Test-set record index", ylabel="Severity class",
            title="Observed vs. predicted severity")
axes[0].legend()

cm = confusion_matrix(y_test_l, logistic_pred, labels=severity_order)
img = axes[1].imshow(cm, cmap="Blues")
axes[1].set(xticks=range(len(severity_order)), yticks=range(len(severity_order)),
            xticklabels=severity_order, yticklabels=severity_order,
            xlabel="Predicted class", ylabel="Actual class",
            title="Confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(img, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

**Interpretation:** Each mismatched marker on the left represents a classification error. In the confusion matrix, off-diagonal values show exactly which severity classes the model mixes up (for example, adjacent Medium and High categories). No Critical classification can be evaluated if the cleaned sample contains zero Critical cases.

## 4. Summary of observed model performance

This cell displays the held-out metrics together and writes a short, data-driven interpretation when the notebook is run. Regression R² and classification accuracy are different quantities and should **not** be compared as if they were the same score.

In [ ]:
results = pd.DataFrame([
    {"Task": "Simple linear regression", "Target": "word_count",
     "Metric": "R²", "Test result": simple_r2},
    {"Task": "Multiple linear regression", "Target": "description_length",
     "Metric": "R²", "Test result": multi_r2},
    {"Task": "Logistic regression", "Target": "severity",
     "Metric": "Accuracy", "Test result": logistic_accuracy},
    {"Task": "Logistic regression", "Target": "severity",
     "Metric": "Macro-F1", "Test result": logistic_macro_f1},
])
display(results.round(4))

display(Markdown(
    f"**Data-driven interpretation.** The simple model achieved held-out R² = "
    f"**{simple_r2:.3f}** predicting word count from character count. "
    f"The four-character-count multiple model achieved R² = **{multi_r2:.3f}**, "
    f"which reflects the arithmetic identity built into its attributes. "
    f"The CVSS logistic model achieved **{logistic_accuracy:.1%} test accuracy** "
    f"and **{logistic_macro_f1:.3f} macro-F1**, compared with a "
    f"**{majority_baseline:.1%} majority-class baseline**. "
    "Its severity classification must be understood as reproducing the CVSS labeling scheme."
))

## 5. Conclusion: what we learned

This report applied three scikit-learn regression techniques to the cleaned UCKG cybersecurity dataset, using an unseen test subset for evaluation and plots to inspect predictions.

- **Simple linear regression:** Character count helps estimate a description's word count. It demonstrates a positive numerical relationship and the roles of R², MAE, RMSE, and residual errors.
- **Multiple linear regression:** Four numerical character-category features reconstruct description length. The exceptionally high fit is expected because the predictors **mathematically determine the target**; this is a useful check of implementation, **not** evidence of generalization to an independent outcome.
- **Logistic regression:** Encoded CVSS vector attributes are used to classify CVE severity. Test accuracy, macro-F1, balanced accuracy, and the confusion matrix reveal how well known severity labels are reproduced. The class imbalance and absent Critical samples limit what can be concluded.

**Overall lesson:** High evaluation scores need context. Mechanical relationships and label-adjacent features can yield excellent results even when the model contributes little new information. For the later graph-machine-learning project, we should train on independently available vulnerability and graph features, avoid direct label leakage from CVSS scoring components, test on new CVEs (ideally with time-based splitting), and seek enough examples of every target severity class.

**Before submission:** Run All against the actual `../dataset/cve_clean.parquet`, inspect the printed metrics and figures, save the executed notebook, and export that executed notebook as `regression.ipynb.pdf` so the submitted PDF contains real results.